# Molecular dynamics — velocity-Verlet + a weak-coupling thermostat

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook runs the same 2-D Lennard-Jones + Coulomb system as the other MD
notebook, but with two changes to the engine:

1. the **velocity-Verlet** integrator in place of the plain (position) Verlet, and
2. a **Berendsen weak-coupling thermostat** that drives the temperature towards a
   target value.

### Velocity-Verlet

Writing the acceleration as $a=F/m$, one step from $t$ to $t+h$ is

$$
\begin{aligned}
r(t+h) &= r(t) + v(t)\,h + \tfrac12\,a(t)\,h^2, \\
&\quad\text{(evaluate the new forces } F(t+h) \text{ at } r(t+h)) \\
v(t+h) &= v(t) + \tfrac12\big[\,a(t)+a(t+h)\,\big]\,h .
\end{aligned}
$$

Unlike the position-Verlet form $r(t+h)=2r(t)-r(t-h)+a(t)h^2$, velocity-Verlet is
**self-starting** (no special first step, no need to remember $r(t-h)$) and gives
**positions and velocities at the same instant** — so the kinetic and potential
energy are evaluated synchronously and the total energy is clean. Each step keeps
the current forces around to move the positions, then recomputes them once to finish
the velocity update (one force evaluation per step, same cost as before).

### Berendsen thermostat (weak coupling)

Plain Verlet conserves energy (an **NVE** ensemble); the temperature is whatever the
initial conditions give. To hold a target temperature $T_0$ (an **NVT**-like
ensemble) we couple the system to an external heat bath and rescale the velocities
each step by

$$\lambda=\sqrt{\,1+\frac{h}{\tau}\left(\frac{T_0}{T}-1\right)}\,,\qquad v\to\lambda v,$$

where $T$ is the current instantaneous temperature and $\tau$ the **coupling time**:

- $\tau=h$ → $\lambda=\sqrt{T_0/T}$: temperature reset to $T_0$ **every step** (strong
  coupling / simple velocity rescaling),
- $\tau\gg h$ → $\lambda\approx1$: **weak coupling**, the temperature relaxes slowly
  and the dynamics stay smooth.

Only stdlib `math`/`random` plus **matplotlib** are needed.

## 1. Imports

In [ ]:
# Colab-friendly install guard (only matplotlib is 3rd-party)
import importlib.util, subprocess, sys
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

import math
import random
import statistics
from math import sqrt, log, sin, cos
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.animation import FuncAnimation

rc('animation', html='jshtml', embed_limit=64)   # MB; keep all animation frames
%matplotlib inline

## 2. Helper functions

In [ ]:
def dist(A, B):
    return math.sqrt((A[0] - B[0]) ** 2 + (A[1] - B[1]) ** 2)

def SignR(a, b):
    # used to pick the nearest periodic image (minimum-image convention)
    return a if b > 0 else -a

def charge_color(charge, qat):
    return "#FFFFFF" if charge == qat else "#333333" 

## 3. Energy functions (LJ + Coulomb, kinetic energy / temperature)

The energy and force loops clamp the pair distance to `SoftCore` (a **soft core**,
defined in §6) — `distsquare = max(distsquare, SoftCore2)` — so a deeply overlapping
pair feels a large but *finite* repulsion instead of the diverging $r^{-12}$ one.
This keeps the long run stable through the occasional hard collision (see §6).

In [ ]:
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1 / distsquare) ** 3 * rmin_exp6
    return epsilon * Z * (Z - 1)

def Coulomb2(distsquare, dielec, qa, qb):
    return qa * qb / (dielec * math.sqrt(distsquare))

def Calc_Ene2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, elec=1):
    Ene = ELJ = ECoul = 0.0
    rmin_exp6 = rmin ** 6
    for i in range(len(coord) - 1):
        for j in range(i + 1, len(coord)):
            distsquare = 0.0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k] / 2
                tmp = tmp - SignR(halfbox, tmp - halfbox) - SignR(halfbox, tmp + halfbox)
                distsquare += tmp ** 2
            if distsquare < cutoffsquare:
                distsquare = max(distsquare, SoftCore2)   # soft core (see section 6)
                qa, qb = coord[i][2], coord[j][2]
                vdw = LJ2(distsquare, epsilon, rmin_exp6); Ene += vdw; ELJ += vdw
                if elec:
                    cc = Coulomb2(distsquare, dielec, qa, qb); Ene += cc; ECoul += cc
    return Ene, ELJ, ECoul

def calc_temp(vel, nat, k, mass):
    v2 = 0.0
    for vx, vy in vel:
        v2 += vx ** 2 + vy ** 2
    kin = 0.5 * mass * v2
    return kin, kin / (nat * k)   # N k T = kinetic energy

## 4. Force functions

In [ ]:
def ForceLJ2(distsquare, epsilon, rmin_exp6, xi):
    rij = math.sqrt(distsquare)
    Z = (1 / distsquare) ** 3 * rmin_exp6
    return epsilon * (2 * Z - 1) * (rmin_exp6 * (-6.0 / rij ** 7)) * (xi / rij)

def ForceCoulomb(distsquare, dielec, qa, qb, xi):
    rij = math.sqrt(distsquare)
    return -1.0 * (qa * qb / dielec) * (1 / distsquare) * (xi / rij)

def Calc_Force(coord, epsilon, rmin, dielec, cutoffsquare, boxdim):
    Force = []
    rmin_exp6 = rmin ** 6
    for i in range(len(coord)):
        tmpforce = [0.0, 0.0]
        for j in range(len(coord)):
            if i == j:
                continue
            distsquare = 0.0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k] / 2
                tmp = tmp - SignR(halfbox, tmp - halfbox) - SignR(halfbox, tmp + halfbox)
                distsquare += tmp ** 2
            if distsquare < cutoffsquare:
                distsquare = max(distsquare, SoftCore2)   # soft core (see section 6)
                qa, qb = coord[i][2], coord[j][2]
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    tmpforce[k] += ForceLJ2(distsquare, epsilon, rmin_exp6, tmp)
                    tmpforce[k] += ForceCoulomb(distsquare, dielec, qa, qb, tmp)
        Force.append(tmpforce)
    return Force

## 5. Velocity-Verlet integrator + Berendsen thermostat

`VelVerlet_Pos` does the position update, `VelVerlet_Vel` finishes the velocity
update from the average of the old and new forces, and `Berendsen` applies the
weak-coupling velocity rescaling.

In [ ]:
def VelVerlet_Pos(coord, vel, force, h, mass):
    # r(t+h) = r(t) + v(t) h + 1/2 (F/m) h^2
    return [[coord[i][0] + vel[i][0]*h + 0.5*force[i][0]/mass*h**2,
             coord[i][1] + vel[i][1]*h + 0.5*force[i][1]/mass*h**2,
             coord[i][2]] for i in range(len(coord))]

def VelVerlet_Vel(vel, force_old, force_new, h, mass):
    # v(t+h) = v(t) + 1/2 (F_old + F_new)/m h
    return [[vel[i][0] + 0.5*(force_old[i][0] + force_new[i][0])/mass*h,
             vel[i][1] + 0.5*(force_old[i][1] + force_new[i][1])/mass*h]
            for i in range(len(vel))]

def Berendsen(vel, temp_current, temp_target, h, tau):
    # weak-coupling velocity rescaling: lambda = sqrt(1 + (h/tau)(T0/T - 1))
    if temp_current <= 0.0:
        return vel
    fac = 1.0 + (h / tau) * (temp_target / temp_current - 1.0)
    lam = math.sqrt(max(fac, 0.0))
    return [[v[0]*lam, v[1]*lam] for v in vel]

## 6. Parameters

The system parameters are identical to the other MD/EM notebooks (same particles,
box and interactions); the second table adds the molecular-dynamics and thermostat
controls. Change any of them and re-run **this cell together with the Initialisation
and Run cells just below** to explore their effect (or use *Kernel → Restart & Run
All*). Values are in the toy model's arbitrary units (lengths in box/canvas units,
energies loosely in kcal/mol, time in the corresponding MD units).

### The molecular system and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms, or initialisation fails |
| `Mass` | particle mass (sets the response to a force, $a = F/m$) | 10 |
| `Rmin` | position of the LJ energy minimum | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth (van der Waals strength) | 1–100 |
| `Dielec` | dielectric constant (electrostatic screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |
| `SoftCore` | soft-core distance: below this separation the repulsion is capped so a rare hard collision cannot blow up the long run (§8) | `0.8 * Rmin` |
| `Seed` | random seed for the initial configuration and velocities (reproducibility) | 100 |

### Molecular dynamics and thermostat

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `Temperature` | target temperature $T_0$ the thermostat drives the system to | 300 K |
| `timestep` | MD integration time step $h$ (smaller = more accurate, more steps) | 1e-3 |
| `tauT` | thermostat coupling time: **small → strong coupling** (fast, forcing); **large → weak coupling** (gentle) | 0.1 |
| `use_thermostat` | thermostat on (**NVT**, temperature held) or off (**NVE**, energy conserved) | True |
| `nsteps` | length of the production run (§8) | 50000 |
| `report_every` | print block-averaged energy/temperature stats every this many steps | 1000 |
| `nsteps_demo` | shorter length used only for the comparison demos (§9, §12) | 4000 |

In [ ]:
# --- system (shared with the EM / MD notebooks) ---
nAtoms  = 20
Radius  = 25.0
Mass    = 10.0
Rmin    = 2.24 * Radius
BoxDim  = [500.0, 500.0]
Epsilon = 25.0
Dielec  = 1.0
qat     = Radius
frac_neg = 0.5
OverlapFr = 0.0
CutOff  = 250.0
CutOffSquare = CutOff ** 2
cstboltz = 1000 * 0.00198722 / 4.18   # Boltzmann constant in kJ/mol/K
Seed    = 100

# --- soft core ---
# Over a long run (50000 steps) an oppositely-charged pair eventually drifts into
# hard contact, and the r^-12 core cannot absorb the collision at this timestep, so
# the energy blows up. A soft core clamps the pair distance to >= SoftCore in the
# energy/force evaluation, capping the repulsion for deeply overlapping pairs and
# keeping the integration stable. It only ever activates on a near head-on collision;
# the short comparison runs (sections 12-13) never trigger it. (The short EM/MD
# notebooks omit it because a few-thousand-step run essentially never collides.)
SoftCore  = 0.8 * Rmin
SoftCore2 = SoftCore ** 2
# (the clamp is not perfectly energy-conserving: it injects a little energy on each
#  collision, which the thermostat absorbs. Section 9 compares NVE/NVT over a short
#  window where no collision fires, so that stays a clean energy-conservation test.)

# --- MD + thermostat controls ---
Temperature   = 300.0    # target temperature (K)
timestep      = 1.0e-3   # MD time step
tauT          = 0.1      # thermostat coupling time (small = strong, large = weak)
use_thermostat = True    # False -> plain NVE (no temperature control)
nsteps        = 50000    # length of the main production run (section 8)
report_every  = 1000     # print energy/temperature stats every this many steps
nsteps_demo   = 4000     # shorter runs used only for the comparison demos (11-13)

## 7. Initialisation (random configuration + Maxwell velocities)

In [ ]:
def InitConf(n, dim, radius, qat, frac_neg, seed):
    random.seed(seed)
    coord = []; nneg = int(n * frac_neg)
    def place(charge):
        ntrial = 0
        while True:
            x = random.random() * (dim[0] - 2*radius) + radius
            y = random.random() * (dim[1] - 2*radius) + radius
            if all(dist(c, [x, y]) >= (1 - OverlapFr) * 2 * radius for c in coord):
                coord.append([x, y, charge]); return
            ntrial += 1
            if ntrial > 100000:
                raise RuntimeError("initialisation failed -> reduce radius or nAtoms")
    for _ in range(nneg):     place(-qat)
    for _ in range(n - nneg): place(+qat)
    return coord

def InitVel(n, temperature, cstboltz, mass, seed=1):
    random.seed(seed)
    stdev = math.sqrt(cstboltz * temperature / mass)
    v = []
    for _ in range(n):
        r1, r2 = random.random(), random.random()
        v.append([math.sqrt(-2*math.log(r1))*math.cos(r2)*stdev,
                  math.sqrt(-2*math.log(r1))*math.sin(0.5*r2)*stdev])
    # remove overall (centre-of-mass) motion
    vxt = sum(a[0] for a in v)/n; vyt = sum(a[1] for a in v)/n
    for a in v: a[0] -= vxt; a[1] -= vyt
    # scale to hit the requested temperature exactly
    _, tt = calc_temp(v, n, cstboltz, mass); sc = math.sqrt(temperature / tt)
    return [[a[0]*sc, a[1]*sc] for a in v]

Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg, Seed)
Velocity   = InitVel(nAtoms, Temperature, cstboltz, Mass)
print(f"placed {len(Atom_Coord)} atoms; initial T = "
      f"{calc_temp(Velocity, nAtoms, cstboltz, Mass)[1]:.1f} K")

## 8. Run the velocity-Verlet MD

The headless run loop (replacing the Tk `Go` callback): compute the initial forces,
then each step do the velocity-Verlet position update → recompute forces → velocity
update → **Berendsen thermostat** → wrap positions into the box. We record the
trajectory and the per-step potential / kinetic / total energy and temperature.

This is the **production run** — `nsteps = 50000` steps. Passing
`report_every = 1000` prints block-averaged energy and temperature statistics
(mean, and standard deviation for $E_\text{tot}$ and $T$) every 1000 steps, so you can
watch the system settle and the thermostat hold the temperature.

In [ ]:
def run_md(coord0, vel0, nsteps, dt, mass, T0, tauT, use_thermostat=True,
           report_every=None):
    coord = [list(c) for c in coord0]
    vel   = [list(v) for v in vel0]
    traj, Epot, Ekin, Etot, Temp, Speed = [], [], [], [], [], []
    Evdw, Ecoul = [], []   # van der Waals (LJ) and electrostatic (Coulomb) parts of Epot
    Force = Calc_Force(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
    if report_every:
        print(f"{'step':>7} | {'<Epot>':>8} {'<Ekin>':>8} | {'<Etot>':>9} {'sd(Etot)':>9}"
              f" | {'<T>':>7} {'sd(T)':>6}  (block averages over {report_every} steps)")
    for it in range(nsteps):
        # velocity-Verlet: move, recompute forces, then finish the velocity update
        coord = VelVerlet_Pos(coord, vel, Force, dt, mass)
        Fnew = Calc_Force(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
        vel = VelVerlet_Vel(vel, Force, Fnew, dt, mass); Force = Fnew
        Kin, T = calc_temp(vel, nAtoms, cstboltz, mass)
        if use_thermostat:
            vel = Berendsen(vel, T, T0, dt, tauT)
            Kin, T = calc_temp(vel, nAtoms, cstboltz, mass)
        Ene, elj, ecoul = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
        # periodic boundary conditions (velocity-Verlet stores velocities explicitly,
        # so wrapping the positions does not corrupt anything)
        for pp in range(len(coord)):
            for i in range(2):
                if coord[pp][i] < 0:         coord[pp][i] += BoxDim[i]
                if coord[pp][i] > BoxDim[i]: coord[pp][i] -= BoxDim[i]
        traj.append([list(c) for c in coord])
        Speed.append([math.hypot(v[0], v[1]) for v in vel])
        Epot.append(Ene); Ekin.append(Kin); Etot.append(Ene + Kin); Temp.append(T)
        Evdw.append(elj); Ecoul.append(ecoul)
        # periodic energy / temperature statistics (block averages)
        if report_every and (it + 1) % report_every == 0:
            a, b = it + 1 - report_every, it + 1
            print(f"{it+1:7d} | {statistics.mean(Epot[a:b]):8.1f} {statistics.mean(Ekin[a:b]):8.1f}"
                  f" | {statistics.mean(Etot[a:b]):9.1f} {statistics.pstdev(Etot[a:b]):9.2f}"
                  f" | {statistics.mean(Temp[a:b]):7.1f} {statistics.pstdev(Temp[a:b]):6.2f}")
    return dict(traj=traj, Epot=Epot, Ekin=Ekin, Etot=Etot, Temp=Temp, Speed=Speed,
                Evdw=Evdw, Ecoul=Ecoul)

out = run_md(Atom_Coord, Velocity, nsteps, timestep, Mass, Temperature, tauT,
             use_thermostat=use_thermostat, report_every=report_every)
traj, Epot, Ekin, Etot, Temp, Speed = (out["traj"], out["Epot"], out["Ekin"],
                                       out["Etot"], out["Temp"], out["Speed"])
Evdw, Ecoul = out["Evdw"], out["Ecoul"]
print(f"\nran {nsteps} steps (velocity-Verlet, thermostat={'on' if use_thermostat else 'off'})")
print(f"overall: <Etot> = {statistics.mean(Etot):.1f}   "
      f"<T> = {statistics.mean(Temp):.1f} K (target {Temperature:.0f} K)")

And here are the recorded energies and temperature over the whole run:

- **left** — the total picture: kinetic, potential and total energy;
- **middle** — the **potential energy split into its two physical parts**, the
  **van der Waals** (Lennard-Jones) term and the **electrostatic** (Coulomb) term,
  whose sum is $E_\text{pot}$;
- **right** — the temperature.

The kinetic energy (hence temperature) is held near the target by the thermostat,
while the potential energy — and its LJ / Coulomb parts — fluctuate as the particles
rearrange. The occasional sharp spike is a hard collision caught by the **soft core**
(§6): instead of blowing up, the pair bounces, the temperature jumps briefly, and the
thermostat brings it straight back.

In [ ]:
t = [i * timestep for i in range(nsteps)]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
# (1) total / kinetic / potential energy
ax[0].plot(t, Ekin, color="#2ca02c", lw=0.5, label="E$_{kin}$")
ax[0].plot(t, Epot, color="#1f77b4", lw=0.5, label="E$_{pot}$")
ax[0].plot(t, Etot, color="#000000", lw=0.8, label="E$_{tot}$")
ax[0].set_xlabel("time"); ax[0].set_ylabel("energy"); ax[0].legend(fontsize=8)
ax[0].set_title(f"Energies ({nsteps}-step run)")
# (2) potential energy broken into van der Waals (LJ) and electrostatic (Coulomb)
ax[1].plot(t, Evdw,  color="#9467bd", lw=0.5, label="van der Waals (LJ)")
ax[1].plot(t, Ecoul, color="#ff7f0e", lw=0.5, label="electrostatic (Coulomb)")
ax[1].plot(t, Epot,  color="#1f77b4", lw=0.8, label="E$_{pot}$ = LJ + Coulomb")
ax[1].axhline(0, color="grey", lw=0.6)
ax[1].set_xlabel("time"); ax[1].set_ylabel("energy"); ax[1].legend(fontsize=8)
ax[1].set_title("Potential energy: vdW vs electrostatic")
# (3) temperature
ax[2].plot(t, Temp, color="#d62728", lw=0.5)
ax[2].axhline(Temperature, color="k", ls="--", lw=1, label=f"target {Temperature:.0f} K")
ax[2].set_xlabel("time"); ax[2].set_ylabel("temperature (K)")
ax[2].legend(fontsize=8); ax[2].set_title("Temperature held by the thermostat")
plt.tight_layout(); plt.show()

print(f"averages over the run:  E_vdW = {statistics.mean(Evdw):7.1f}   "
      f"E_elec = {statistics.mean(Ecoul):7.1f}   E_pot = {statistics.mean(Epot):7.1f}")

## 9. Why a thermostat does *not* conserve $E_\text{tot}$

A thermostat and energy conservation pull in opposite directions, and it is worth
seeing why. Run the **same** system twice: once **with** the thermostat (NVT) and
once **without** it (NVE, plain energy-conserving velocity-Verlet).

Two things to notice in the plots below:

- **Without a thermostat (NVE, grey):** the **total energy is flat** — velocity-Verlet
  conserves it to a fraction of a percent. But the *temperature* wanders (grey, right),
  because kinetic and potential energy trade back and forth ($E_\text{kin}\!\downarrow$
  when $E_\text{pot}\!\uparrow$ and vice-versa).
- **With the thermostat (NVT, coloured):** the **temperature is held** at the target
  (right), but the **total energy is no longer conserved** (left) — the thermostat pins
  the kinetic energy, so as the potential energy fluctuates the bath must add or remove
  energy, and $E_\text{tot}=E_\text{kin}+E_\text{pot}$ moves with $E_\text{pot}$.

So the wobble you see in $E_\text{tot}$ under the thermostat is **not** an integrator
error — it is the heat bath doing its job. (Here $E_\text{kin}\approx2850 \gg
|E_\text{pot}|\approx200$, so on this scale the $E_\text{tot}$ swing is small; the clean
energy conservation of velocity-Verlet itself is the flat grey NVE curve. This comparison
uses a short window so the soft core never fires — it is a tiny energy *source* on a
collision, which the thermostat quietly absorbs but which would slowly heat an
unthermostatted run.)

In [ ]:
# Run the same system twice over a short window (no collision -> soft core never
# fires, so this isolates the NVT-vs-NVE difference cleanly).
out_nvt = run_md(Atom_Coord, Velocity, nsteps_demo, timestep, Mass, Temperature, tauT,
                 use_thermostat=True)
out_nve = run_md(Atom_Coord, Velocity, nsteps_demo, timestep, Mass, Temperature, tauT,
                 use_thermostat=False)
t = [i * timestep for i in range(nsteps_demo)]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
# total energy
ax[0].plot(t, out_nve["Etot"], color="#999999", lw=1.4, label="no thermostat (NVE)")
ax[0].plot(t, out_nvt["Etot"], color="#1f77b4", lw=1.0, label="thermostat (NVT)")
ax[0].set_xlabel("time"); ax[0].set_ylabel("total energy")
ax[0].legend(fontsize=8); ax[0].set_title("Total energy: conserved only without the thermostat")
# temperature
ax[1].plot(t, out_nve["Temp"], color="#999999", lw=1.0, label="no thermostat (NVE)")
ax[1].plot(t, out_nvt["Temp"], color="#d62728", lw=1.0, label="thermostat (NVT)")
ax[1].axhline(Temperature, color="k", ls="--", lw=1, label=f"target {Temperature:.0f} K")
ax[1].set_xlabel("time"); ax[1].set_ylabel("temperature (K)")
ax[1].legend(fontsize=8); ax[1].set_title("Temperature: held only with the thermostat")
plt.tight_layout(); plt.show()

def drift(E): return 100*abs(E[-1]-E[0])/abs(E[0])
print(f"NVE (no thermostat): Etot drift {drift(out_nve['Etot']):.2f}%   "
      f"T range {min(out_nve['Temp']):.0f}-{max(out_nve['Temp']):.0f} K")
print(f"NVT (thermostat)   : Etot drift {drift(out_nvt['Etot']):.2f}%   "
      f"<T> {sum(out_nvt['Temp'])/len(out_nvt['Temp']):.1f} K (target {Temperature:.0f})")

## 10. Configuration (initial vs final) and animation

In [ ]:
from matplotlib.patches import Circle, Rectangle
fig, ax = plt.subplots(1, 2, figsize=(9.5, 5))
for a, coords, title in ((ax[0], traj[0], "initial"), (ax[1], traj[-1], f"after {nsteps} steps")):
    a.add_patch(Rectangle((0, 0), BoxDim[0], BoxDim[1], fc="#ccddff", ec="k", lw=1.2))
    for c in coords:
        a.add_patch(Circle((c[0], c[1]), Radius, fc=charge_color(c[2], qat), ec="k", lw=0.5))
    a.set_xlim(-10, BoxDim[0]+10); a.set_ylim(-10, BoxDim[1]+10); a.set_aspect("equal")
    a.set_xticks([]); a.set_yticks([]); a.set_title(title, fontsize=10)
fig.suptitle("velocity-Verlet MD  (white +, dark −)", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
stride = max(1, nsteps // 120)
frames = list(range(0, nsteps, stride))
fig, axa = plt.subplots(figsize=(5.4, 5.4))
axa.set_xlim(0, BoxDim[0]); axa.set_ylim(0, BoxDim[1]); axa.set_aspect("equal")
axa.set_facecolor("#ccddff"); axa.set_xticks([]); axa.set_yticks([])
cols = [charge_color(c[2], qat) for c in traj[0]]
scat = axa.scatter([c[0] for c in traj[0]], [c[1] for c in traj[0]],
                   s=520, c=cols, edgecolors="k")
ttl = axa.set_title("")

def update(n):
    scat.set_offsets([[c[0], c[1]] for c in traj[n]])
    ttl.set_text(f"step {n}   T = {Temp[n]:.0f} K   fastest atom = {max(Speed[n]):.1f}")
    return scat,
anim = FuncAnimation(fig, update, frames=frames, interval=60, blit=False)
plt.close(fig); anim

## 11. Temperature is an *average* — individual speeds vary a lot

Watching the animation you may notice individual particles darting about quickly
while the temperature in the title barely moves — it looks like the temperature
"should" be higher. It is worth understanding why this is correct and **not** a bug
in the temperature calculation.

**Temperature is not how fast any one particle looks.** It is the *average* kinetic
energy of the whole system: from $\sum_i \tfrac12 m v_i^2 = N k_B T$ in 2-D,

$$T=\frac{1}{N k_B}\sum_i \tfrac12 m v_i^2 = \frac{m}{2 k_B}\,\langle v^2\rangle .$$

So `calc_temp` is exactly proportional to the **mean of $v^2$** over all particles.
At any instant the individual speeds follow a broad **Maxwell–Boltzmann**
distribution — most particles near the average, some almost stationary, a few moving
2–3× faster. Those few fast ones catch the eye in the animation, but they barely
shift the *average*, so the temperature is steady. On top of that, the **thermostat
pins the average**, damping even the modest global fluctuations. The plots below make
this concrete: the fastest atom is $\sim$2–3× the mean at every step, yet the mean
(hence T) hardly moves.

In [ ]:
# pool individual speeds over the whole run for the distribution
allspeeds = [s for frame in Speed for s in frame]
mean_sp = [sum(f)/len(f) for f in Speed]
max_sp  = [max(f) for f in Speed]
min_sp  = [min(f) for f in Speed]
t = [i * timestep for i in range(nsteps)]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
# (A) Maxwell-Boltzmann speed distribution
ax[0].hist(allspeeds, bins=40, color="#1f77b4", edgecolor="white", density=True)
mbar = sum(allspeeds)/len(allspeeds)
ax[0].axvline(mbar, color="k", ls="--", lw=1.2, label=f"mean speed = {mbar:.1f}")
ax[0].set_xlabel("individual particle speed"); ax[0].set_ylabel("probability density")
ax[0].set_title("Maxwell–Boltzmann: a broad spread of speeds"); ax[0].legend(fontsize=8)
# (B) fastest / mean / slowest speed vs time
ax[1].plot(t, max_sp,  color="#d62728", lw=0.8, label="fastest atom")
ax[1].plot(t, mean_sp, color="#000000", lw=1.5, label="mean speed  (∝ √T)")
ax[1].plot(t, min_sp,  color="#2ca02c", lw=0.8, label="slowest atom")
ax[1].set_xlabel("time"); ax[1].set_ylabel("speed")
ax[1].set_title("Individuals vary; the mean (temperature) stays put"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"over the whole run: slowest atom ~{min(min_sp):.1f}, fastest atom ~{max(max_sp):.1f}, "
      f"mean speed {mbar:.1f} (fastest is {max(max_sp)/mbar:.1f}x the mean)")
print(f"temperature stays at {sum(Temp)/len(Temp):.0f} K throughout — it is the average, "
      f"not the fastest particle")

## 12. The thermostat at work — weak vs strong coupling

To see the thermostat control the temperature, we start the **same** system with the
velocities scaled up (a hot start, $T\approx1200$ K) and let the thermostat pull it
back to the 300 K target, for several coupling times `tauT`. Small `tauT` (strong
coupling) snaps the temperature back almost immediately; large `tauT` (weak coupling)
lets it relax gently over many steps. For reference, the **NVE** curve
(`use_thermostat=False`) keeps whatever temperature it started with.

In [ ]:
# hot start: same config, velocities x2  ->  T ~ 4x ~ 1200 K
Vel_hot = [[2*v[0], 2*v[1]] for v in Velocity]

plt.figure(figsize=(9.5, 4.4))
t = [i * timestep for i in range(nsteps_demo)]
# NVE reference (no thermostat)
r_nve = run_md(Atom_Coord, Vel_hot, nsteps_demo, timestep, Mass, Temperature, tauT,
               use_thermostat=False)
plt.plot(t, r_nve["Temp"], color="#999999", lw=1.2, ls=":",
         label="no thermostat (NVE)")
for tau, col in ((0.02, "#d62728"), (0.1, "#ff7f0e"), (0.5, "#1f77b4")):
    r = run_md(Atom_Coord, Vel_hot, nsteps_demo, timestep, Mass, Temperature, tau,
               use_thermostat=True)
    reach = next((i for i, T in enumerate(r["Temp"]) if abs(T - Temperature) < 30), None)
    plt.plot(t, r["Temp"], color=col, lw=1.3,
             label=f"tauT = {tau}  (T≈target after {reach} steps)")
plt.axhline(Temperature, color="k", ls="--", lw=1)
plt.xlabel("time"); plt.ylabel("temperature (K)")
plt.title("Berendsen thermostat: relaxation of a hot start to 300 K")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 13. Take-home messages

- **Velocity-Verlet** propagates positions and velocities together: it is
  self-starting, needs no previous-step positions, and gives synchronised
  $r$ and $v$ so energies are evaluated at the same instant. It costs one force
  evaluation per step, exactly like plain Verlet, and conserves energy just as well.
- With **no thermostat** the dynamics are **NVE** (constant energy); the temperature
  is whatever the initial velocities set.
- A **Berendsen weak-coupling thermostat** nudges the velocities each step so the
  temperature relaxes to a target $T_0$ with a time constant `tauT`. **Small `tauT`
  = strong coupling** (fast, but perturbs the dynamics — in the limit it is simple
  velocity rescaling); **large `tauT` = weak coupling** (gentle, slow relaxation).
- **Temperature is a whole-system average**, $T\propto\langle v^2\rangle$, not the
  speed of the fastest-looking particle. Individual speeds follow a broad
  Maxwell–Boltzmann distribution (some slow, a few 2–3× the mean), so the average — and
  hence T — stays steady even when single particles dart about (§11).
- Berendsen is simple and robust for *reaching* and *holding* a temperature, but it
  does not reproduce the exact canonical (NVT) velocity distribution — for rigorous
  NVT sampling one uses a Nosé–Hoover or stochastic (Langevin / velocity-rescale)
  thermostat. For this teaching model it is exactly what we want.